# FeatureImpact — Notebook 5: Bayesian A/B Testing

**Objective:** Answer the question frequentist tests cannot: *What is the probability that treatment is better than control?*

Bayesian testing gives us:
- P(treatment > control) — a direct, intuitive probability
- Expected Loss — the cost of making the wrong decision
- Credible intervals instead of confidence intervals
- No sample size requirements — we can update beliefs as data arrives

We use the **Beta-Binomial model** which is analytically tractable and perfect for conversion rates.

> Note: We implement Bayesian inference using SciPy's Beta distribution (no PyMC required), making this notebook fully portable.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
np.random.seed(42)

# Load data
df = pd.read_csv("data/ab_experiment_raw.csv")
control   = df[df["group"] == "control"]
treatment = df[df["group"] == "treatment"]

# Observed data
n_ctrl    = len(control)
n_trt     = len(treatment)
conv_ctrl = control["converted"].sum()
conv_trt  = treatment["converted"].sum()
fail_ctrl = n_ctrl - conv_ctrl
fail_trt  = n_trt  - conv_trt

print("Observed Data")
print(f"  Control:   {conv_ctrl} conversions / {n_ctrl} users ({conv_ctrl/n_ctrl:.2%})")
print(f"  Treatment: {conv_trt} conversions / {n_trt} users ({conv_trt/n_trt:.2%})")

## 2. The Beta-Binomial Model

**Prior:** Beta(1, 1) — uniform, no prior knowledge about conversion rate.

**Likelihood:** Binomial — number of conversions given a true conversion rate.

**Posterior:** Beta(alpha + successes, beta + failures) — analytically derived.

This is the beauty of conjugate priors: no MCMC needed.

In [ ]:
# Prior parameters (uninformative: Beta(1,1) = Uniform)
prior_alpha = 1
prior_beta  = 1

# Posterior parameters
post_alpha_ctrl = prior_alpha + conv_ctrl
post_beta_ctrl  = prior_beta  + fail_ctrl
post_alpha_trt  = prior_alpha + conv_trt
post_beta_trt   = prior_beta  + fail_trt

# Posterior distributions
posterior_ctrl = stats.beta(post_alpha_ctrl, post_beta_ctrl)
posterior_trt  = stats.beta(post_alpha_trt,  post_beta_trt)

print("Posterior Distribution Parameters")
print(f"  Control   : Beta({post_alpha_ctrl}, {post_beta_ctrl})")
print(f"  Treatment : Beta({post_alpha_trt}, {post_beta_trt})")
print(f"\n  Control posterior mean   : {posterior_ctrl.mean():.4f}")
print(f"  Treatment posterior mean : {posterior_trt.mean():.4f}")

## 3. Visualise Prior vs Posterior

In [ ]:
x     = np.linspace(0.05, 0.25, 1000)
prior = stats.beta(prior_alpha, prior_beta)

palette = {"control": "#4C9BE8", "treatment": "#F4845F"}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Prior vs Posterior Distributions (Beta-Binomial Model)",
             fontsize=13, fontweight="bold")

for ax, (group_label, post, n_obs, conv) in zip(axes, [
    ("control",   posterior_ctrl, n_ctrl, conv_ctrl),
    ("treatment", posterior_trt,  n_trt,  conv_trt),
]):
    color = palette[group_label]
    ci_low, ci_high = post.ppf(0.025), post.ppf(0.975)

    ax.plot(x, prior.pdf(x), color="grey", linestyle="--",
            linewidth=1.5, label="Prior Beta(1,1)")
    ax.fill_between(x, post.pdf(x), alpha=0.3, color=color)
    ax.plot(x, post.pdf(x), color=color, linewidth=2.5, label="Posterior")
    ax.axvline(ci_low,  color=color, linestyle=":", linewidth=1.5)
    ax.axvline(ci_high, color=color, linestyle=":", linewidth=1.5)
    ax.axvline(post.mean(), color=color, linestyle="-", linewidth=1.5, alpha=0.6)

    ax.set_title(f"{group_label.capitalize()}: {conv}/{n_obs} conversions",
                 fontweight="bold")
    ax.set_xlabel("Conversion Rate (p)")
    ax.set_ylabel("Density")
    ax.legend()
    ax.text(0.97, 0.95,
            f"Mean: {post.mean():.3f}\n95% CI: [{ci_low:.3f}, {ci_high:.3f}]",
            transform=ax.transAxes, ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.savefig("nb5_prior_posterior.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. P(Treatment > Control) via Monte Carlo Sampling

In [ ]:
N_SAMPLES = 100_000

samples_ctrl = posterior_ctrl.rvs(N_SAMPLES)
samples_trt  = posterior_trt.rvs(N_SAMPLES)

p_trt_wins          = np.mean(samples_trt > samples_ctrl)
lift_samples        = samples_trt - samples_ctrl
expected_lift       = lift_samples.mean()
ci_lift_low         = np.percentile(lift_samples, 2.5)
ci_lift_high        = np.percentile(lift_samples, 97.5)
loss_if_choose_trt  = np.mean(np.maximum(samples_ctrl - samples_trt, 0))
loss_if_choose_ctrl = np.mean(np.maximum(samples_trt  - samples_ctrl, 0))

print("=" * 60)
print("BAYESIAN TEST RESULTS")
print("=" * 60)
print(f"  P(Treatment > Control)        : {p_trt_wins:.2%}")
print(f"  Expected lift                 : {expected_lift:+.4f} ({expected_lift/samples_ctrl.mean():+.1%} relative)")
print(f"  95% Credible Interval (lift)  : [{ci_lift_low:.4f}, {ci_lift_high:.4f}]")
print(f"  Expected Loss (choose TRT)    : {loss_if_choose_trt:.6f}")
print(f"  Expected Loss (choose CTRL)   : {loss_if_choose_ctrl:.6f}")
print("-" * 60)
decision = "SHIP TREATMENT" if loss_if_choose_trt < loss_if_choose_ctrl else "KEEP CONTROL"
print(f"  Decision (min expected loss)  : {decision}")

## 5. Posterior Distribution of the Lift

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Bayesian Results: Posterior Distributions",
             fontsize=13, fontweight="bold")

# Overlapping posteriors
x = np.linspace(0.05, 0.22, 1000)
for label, post, color in [
    ("Control",   posterior_ctrl, "#4C9BE8"),
    ("Treatment", posterior_trt,  "#F4845F")
]:
    axes[0].fill_between(x, post.pdf(x), alpha=0.35, color=color)
    axes[0].plot(x, post.pdf(x), color=color, linewidth=2.5, label=label)

axes[0].set_title("Posterior Distributions Overlaid")
axes[0].set_xlabel("Conversion Rate")
axes[0].set_ylabel("Density")
axes[0].legend()

# Lift distribution
axes[1].hist(lift_samples, bins=80, color="#9B59B6",
             edgecolor="white", alpha=0.7, density=True)
axes[1].axvline(0, color="black", linewidth=1.5,
                linestyle="--", label="No lift")
axes[1].axvline(expected_lift, color="#9B59B6", linewidth=2,
                label=f"Expected lift: {expected_lift:+.3f}")
axes[1].axvline(ci_lift_low,  color="grey", linewidth=1, linestyle=":")
axes[1].axvline(ci_lift_high, color="grey", linewidth=1,
                linestyle=":", label="95% credible interval")

axes[1].set_title(f"Posterior of Lift  |  P(TRT > CTRL) = {p_trt_wins:.1%}")
axes[1].set_xlabel("Lift (Treatment - Control)")
axes[1].set_ylabel("Density")
axes[1].legend()

plt.tight_layout()
plt.savefig("nb5_lift_posterior.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Frequentist vs Bayesian Comparison

In [ ]:
comparison = {
    "Aspect":            ["Core output",    "Interpretation",      "Interval",            "Sample size",        "Business language"],
    "Frequentist":       ["p-value",         "Reject/fail H0",      "Confidence interval", "Required upfront",   "Harder (jargon)"],
    "Bayesian":          ["P(TRT > CTRL)",   "Probability better",  "Credible interval",   "Update as data in",  "92% chance its better"],
}
print(pd.DataFrame(comparison).to_string(index=False))

## Summary

| Finding | Value |
|---|---|
| P(Treatment > Control) | See output above |
| Expected absolute lift | See output above |
| 95% Credible Interval  | See output above |
| Decision               | Ship treatment (lower expected loss) |

**Next:** Notebook 6 — Sequential Testing and Multiple Comparisons.